In [1]:
#imports
import sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from collections import Counter

In [ ]:
file_path = "paste your file path here"

In [ ]:
feature_length = 4096  # all features have this length

feature_description = {
    'tmmx': tf.io.FixedLenFeature([feature_length], tf.float32),
    'FireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'PrevFireMask': tf.io.FixedLenFeature([feature_length], tf.float32),
    'th': tf.io.FixedLenFeature([feature_length], tf.float32),
    'erc': tf.io.FixedLenFeature([feature_length], tf.float32),
    'vs': tf.io.FixedLenFeature([feature_length], tf.float32),
    'elevation': tf.io.FixedLenFeature([feature_length], tf.float32),
    'tmmn': tf.io.FixedLenFeature([feature_length], tf.float32),
    'NDVI': tf.io.FixedLenFeature([feature_length], tf.float32),
    'sph': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pdsi': tf.io.FixedLenFeature([feature_length], tf.float32),
    'pr': tf.io.FixedLenFeature([feature_length], tf.float32),
    'population': tf.io.FixedLenFeature([feature_length], tf.float32),
}

def _parse_function(proto):
    return tf.io.parse_single_example(proto, feature_description)

dataset = tf.data.TFRecordDataset(file_path)
parsed_dataset = dataset.map(_parse_function)

# Define the list of feature names and the label name
features_to_extract = ['tmmx', 'PrevFireMask', 'th', 'erc', 'vs', 'elevation',
                       'tmmn', 'NDVI', 'sph', 'pdsi', 'pr', 'population']
label_to_extract = 'FireMask'

In [ ]:
all_features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
all_labels_data = []

for parsed_record in parsed_dataset:
    #the mean thing is kinda usless, but i don't feel like removing it rn
    if parsed_record['FireMask'].numpy().mean() >= 0:
        for i in range(len(features_to_extract)):
            all_features_data[i].append(
                parsed_record[features_to_extract[i]].numpy().reshape(64, 64)
            )
        all_labels_data.append(parsed_record[label_to_extract].numpy().reshape(64, 64))

all_features_data = np.array(all_features_data)       # (12, 850, 64, 64)
X_transposed = all_features_data.transpose(1, 2, 3, 0)  # (850, 64, 64, 12)
y_grid = np.array(all_labels_data)                    # (850, 64, 64)

n_records = X_transposed.shape[0]
X_neighbors = []
y_neighbors = []

for rec in range(n_records):
    # Exclude 1-pixel border: iterate rows 1..62, cols 1..62
    for r in range(1, 63):
        for c in range(1, 63):
            label = y_grid[rec, r, c]
            if label == -1:
                continue  # skip no-data pixels

            # Extract 3x3 neighborhood for all 12 features, then flatten
            # Shape: (3, 3, 12) -> (108,)
            neighborhood = X_transposed[rec, r-1:r+2, c-1:c+2, :]
            X_neighbors.append(neighborhood.flatten())
            y_neighbors.append(label)

X = np.array(X_neighbors)  # shape: (n_valid_pixels, 108)
y = np.array(y_neighbors)  # shape: (n_valid_pixels,)

assert set(np.unique(y)).issubset({0, 1}), "Found unexpected labels!"
print(f"Shape of features (X): {X.shape}")  # e.g. (n, 108)
print(f"Shape of labels (y): {y.shape}")
print(f"Label distribution: {np.bincount(y.astype(int))}")

In [ ]:
# Full dataset
#Splits the data into training and testing sets
# Assuming X contains features and y contains target variable (labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
#actual model
lr_model = LogisticRegression(random_state=42)
#lr_model = MultiOutputClassifier(lr_estimator)
lr_model.fit(X_train, y_train)
# Make predictions on the test set
predictions = lr_model.predict(X_test)

print('success')

Visualization & analysis

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

overall_accuracy = accuracy_score(y_test, predictions)

overall_precision = precision_score(y_test, predictions, average = 'macro')
overall_recall = recall_score(y_test, predictions, average = 'macro')
overall_f1 = f1_score(y_test, predictions, average = 'macro')

print(f"Accuracy: {overall_accuracy}")

print(f"Overall Precision: {overall_precision}")
print(f"Overall Recall: {overall_recall}")
print(f"Overall F1 Score: {overall_f1}")

print("\nClassification Report:\n", classification_report(y_test, predictions))

In [ ]:
#Single record prediction visualization

dataset = tf.data.TFRecordDataset(file_path)

num_records_to_skip = 2

second_parsed_dataset = dataset.map(_parse_function).skip(num_records_to_skip).take(1)

features_data = [[],[],[],[],[],[],[],[],[],[],[],[]]
labels_data = []

for parsed_record in second_parsed_dataset:
  #if parsed_record['FireMask'].numpy().mean() >= 0:
    for i in range(len(features_to_extract)):
        features_data[i].append(parsed_record[features_to_extract[i]].numpy())
    labels_data.append(parsed_record[label_to_extract].numpy())

features_data = np.array(features_data)

print(f"Shape of features_data: {features_data.shape}") 

# Shape: (n_records, 64, 64, 12)
X_transposed_single = features_data.transpose(1, 2, 0).reshape(-1, 64, 64, 12)
# Shape: (n_records, 64, 64)
y_grid_single = np.array(labels_data).reshape(-1, 64, 64)

n_records = X_transposed_single.shape[0]
X_neighbors_single = []
y_neighbors_single = []

for rec in range(n_records):
    # Exclude 1-pixel border: iterate rows 1..62, cols 1..62
    for r in range(1, 63):
        for c in range(1, 63):
            label = y_grid_single[rec, r, c]

            # Extract 3x3 neighborhood for all 12 features, then flatten
            # Shape: (3, 3, 12) -> (108,)
            neighborhood_single = X_transposed_single[rec, r-1:r+2, c-1:c+2, :]
            X_neighbors_single.append(neighborhood_single.flatten())
            y_neighbors_single.append(label)

X_single = np.array(X_neighbors_single)  # shape: (n_valid_pixels, 108)
y_single = np.array(y_neighbors_single)  # shape: (n_valid_pixels,)

#assert set(np.unique(y_single)).issubset({0, 1}), "Found unexpected labels!"
print(f"Shape of features (X): {X_single.shape}")  # e.g. (n, 108)
print(f"Shape of labels (y): {y_single.shape}")
#print(f"Label distribution: {np.bincount(y_single.astype(int))}")

In [ ]:
single_record_prediction = lr_model.predict(X_single)

plt.figure(figsize=(10, 5))

plt.subplot(1,2,1)
plt.title(f"Prediction of FireMask")
plt.imshow(single_record_prediction.reshape(62,62), cmap="Reds")
plt.colorbar()

plt.subplot(1,2,2)
plt.title(f"Single Record FireMask")
plt.imshow(y_single.reshape(62,62), cmap="Reds")
plt.colorbar()

plt.show()